# Roots and extrema

This section describes how to compute the roots and extrema of a function represented as a Chebyshev series.

## Theory

The roots of the truncated Chebyshev series
```{math}
f(x) = \sum_{n=0}^{N} c_n T_n(x)
```
are found as the eigenvalues of its *colleague matrix* {cite:p}`boyd01`.
This is the Chebyshev analogue of the companion matrix used to find polynomial roots from monomial coefficients.
In terms of the mapped coordinate $\xi \in [-1,1]$, the colleague matrix is
```{math}
:label: colleague
\mat{C} = \begin{pmatrix}
0 & 1 & & & \\
\tfrac{1}{2} & 0 & \tfrac{1}{2} & & \\
 & \ddots & \ddots & \ddots & \\
 & & \tfrac{1}{2} & 0 & \tfrac{1}{2} \\
-\dfrac{c_0}{2c_N} & \cdots & \cdots & \tfrac{1}{2}-\dfrac{c_{N-2}}{2c_N} & -\dfrac{c_{N-1}}{2c_N}
\end{pmatrix} \;.
```
This is an $N \times N$ matrix whose eigenvalues are exactly the roots of $f$ in the $\xi$ variable, real or complex.
For a real valued function, only the real eigenvalues that fall within $[-1, 1]$ are kept and mapped back to $[a, b]$.

The extrema of $f$ are simply the roots of its derivative $f'$.

## Python API

We will use the function $f(x)=\sin(3x)\exp(-0.2x)$ on the interval $[0,5]$ as an example.

In [ ]:
import numpy as np
from cheby import RealFunction

def f_exact(x):
    return np.sin(3 * x) * np.exp(-0.2 * x)

f = RealFunction(f_exact, 0.0, 5.0)

The roots and extrema of a `RealFunction` can be computed with the `roots()` and `extrema()` methods, respectively.

In [ ]:
roots = f.roots()
extrema = f.extrema()

We can plot the function and its roots and extrema to verify that the roots are indeed where the function crosses zero, and the extrema are where the derivative is zero.

In [ ]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_formats = ["svg", "pdf"]

x = np.linspace(0.0, 5.0, 400)
plt.figure()
plt.plot(x, f(x))
plt.plot(roots, f(roots), 'o', label='roots')
plt.plot(extrema, f(extrema), 's', label='extrema')
plt.axhline(0, color='gray', linewidth=0.5)
plt.legend()
plt.xlabel(r'$x$')
plt.show()

As a check, the function values at the roots are close to zero, and the derivative values at the extrema are close to zero:

In [ ]:
print('max |f(root)|:', np.max(np.abs(f(roots))))
print("max |f'(extremum)|:", np.max(np.abs(f.derivative()(extrema))))

In terms of implementation, an important step before the eigenvalues are extracted is that the colleague matrix is *balanced* (a diagonal similarity transform that equalises the row and column norms) to improve the numerical conditioning of the eigenvalue computation for high-order series.

The colleague matrix itself is available through the `colleague()` method, for instance to plot the distribution of all its eigenvalues (not just the real ones inside $[-1, 1]$):

In [ ]:
C = f.colleague()
eigvals = np.linalg.eigvals(C)

plt.figure()
plt.plot(eigvals.real, eigvals.imag, 'o', markersize=3)
theta = np.linspace(0, 2 * np.pi, 200)
plt.plot(np.cos(theta), np.sin(theta), '--', color='gray')
plt.xlabel('Re(x)')
plt.ylabel('Im(x)')
plt.axis('equal')
plt.title('Eigenvalues of the colleague matrix')
plt.show()